In [ ]:
# Locate the repository when opened from any experiment subdirectory.
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents)
             if (p / "smoke_test.py").is_file()
             and (p / "configs/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the PGAE-tau repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Repository: {ROOT}")


In [ ]:
import sys
import os
import time
import importlib
import numpy as np
import pandas as pd
import torch
import joblib
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
from openpyxl.utils import get_column_letter

# ── Path setup ──
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Suppress matplotlib font cache warnings during imports
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib_cache'

# ── Case definitions ──
CASE_CONFIGS = {
    'case14': {
        'sample_length': 6,
        'feature_size': 68,
        'n_bus': 14,
        'lattent_dim': 16,
        'no_layer': 3,
        'gnn_K': 2,
        'gnn_hidden': 16,
        'gnn_latent': 2,
        'anomtrans_d_model': 64,
        'anomtrans_n_heads': 4,
        'anomtrans_e_layers': 2,
        'anomtrans_d_ff': 256,
        'batch_size': 32,
    },
    'case30': {
        'sample_length': 6,
        'feature_size': 142,
        'n_bus': 30,
        'lattent_dim': 32,
        'no_layer': 3,
        'gnn_K': 2,
        'gnn_hidden': 64,
        'gnn_latent': 5,
        'anomtrans_d_model': 128,
        'anomtrans_n_heads': 4,
        'anomtrans_e_layers': 3,
        'anomtrans_d_ff': 512,
        'batch_size': 32,
    },
    'case57': {
        'sample_length': 6,
        'feature_size': 274,
        'n_bus': 57,
        'lattent_dim': 32,
        'no_layer': 3,
        'gnn_K': 3,
        'gnn_hidden': 64,
        'gnn_latent': 8,
        'anomtrans_d_model': 128,
        'anomtrans_n_heads': 4,
        'anomtrans_e_layers': 3,
        'anomtrans_d_ff': 512,
        'batch_size': 32,
    },
}

MODEL_NAMES_DISPLAY = [
    'OCSVM', 'ConvAE', 'LSTMAE', 'GraphAE', 'AT', 'PIConv', 'PGAE-tau'
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Target device: {device}')
print(f'Cases to benchmark: {list(CASE_CONFIGS.keys())}')
print(f'Models: {MODEL_NAMES_DISPLAY}')


In [ ]:
def set_case_config(case_name):
    """Update shared configuration dictionaries for this profiling notebook."""
    cfg = CASE_CONFIGS[case_name]
    
    # Patch configs.config.sys_config
    import configs.config as config_mod
    target = dict(config_mod.sys_config_template[case_name])
    config_mod.sys_config.clear()
    config_mod.sys_config.update(target)
    
    # Patch configs.nn_setting.nn_setting
    import configs.nn_setting as nn_mod
    nn_mod.nn_setting.update({
        'sample_length': cfg['sample_length'],
        'feature_size': cfg['feature_size'],
        'lattent_dim': cfg['lattent_dim'],
        'no_layer': cfg['no_layer'],
        'gnn_K': cfg['gnn_K'],
        'gnn_hidden': cfg['gnn_hidden'],
        'gnn_latent': cfg['gnn_latent'],
        'anomtrans_d_model': cfg['anomtrans_d_model'],
        'anomtrans_n_heads': cfg['anomtrans_n_heads'],
        'anomtrans_e_layers': cfg['anomtrans_e_layers'],
        'anomtrans_d_ff': cfg['anomtrans_d_ff'],
        'batch_size': cfg['batch_size'],
        'device': str(device),
    })

    # Clear cached model modules so they re-read config on next import
    for mod_name in list(sys.modules.keys()):
        if mod_name.startswith('models.') or mod_name.startswith('data.loaders'):
            del sys.modules[mod_name]
    
    print(f'  Switched to {case_name}: feature_size={cfg["feature_size"]}, n_bus={cfg["n_bus"]}')


def profile_torch_model(model_class, model_name, input_shape, device, num_iters=300):
    """Profile a PyTorch model: parameter count and inference time."""
    print(f'    Profiling {model_name}...', end=' ')
    
    try:
        model = model_class().to(device)
    except Exception as e:
        print(f'FAILED: {e}')
        return None
    
    model.eval()
    
    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    
    dummy_input = torch.randn(*input_shape).to(device)
    
    # Warm-up
    with torch.no_grad():
        for _ in range(20):
            _ = model(dummy_input)
    
    # Timed run
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(num_iters):
            _ = model(dummy_input)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.time() - start
    
    avg_time_ms = (elapsed / num_iters) * 1000
    
    print(f'{total_params:.3f}M params, {avg_time_ms:.4f} ms')
    
    return {
        'Total Params (M)': round(total_params, 4),
        'Trainable Params (M)': round(trainable_params, 4),
        'Inference Time (ms)': round(avg_time_ms, 4),
    }


def profile_ocsvm(case_name, num_iters=300):
    """Profile OCSVM (sklearn): support vectors and inference time."""
    pkl_path = f'saved_model/{case_name}/oc_svm_baseline.pkl'
    cfg = CASE_CONFIGS[case_name]
    input_dim = cfg['sample_length'] * cfg['feature_size']
    
    print(f'    Profiling OCSVM ({case_name})...', end=' ')
    
    if not os.path.exists(pkl_path):
        print(f'Model file not found: {pkl_path}')
        return None
    
    try:
        model = joblib.load(pkl_path)
    except Exception as e:
        print(f'FAILED to load: {e}')
        return None
    
    n_support = model.support_vectors_.shape[0]
    
    # Generate test data
    X_test = np.random.randn(200, input_dim).astype(np.float32)
    
    # Warm-up
    for _ in range(20):
        _ = model.decision_function(X_test)
    
    # Timed run
    start = time.time()
    for _ in range(num_iters):
        _ = model.decision_function(X_test)
    elapsed = time.time() - start
    
    avg_time_ms = (elapsed / num_iters) * 1000
    
    print(f'{n_support} SVs, {avg_time_ms:.4f} ms')
    
    return {
        'Total Params (M)': n_support,  # use SV count as "complexity" proxy
        'Trainable Params (M)': n_support,
        'Inference Time (ms)': round(avg_time_ms, 4),
    }


In [ ]:
# ── Iterate over all cases and models ──
all_results = {}  # {(case, model): {metrics...}}

for case_name in ['case14', 'case30', 'case57']:
    print(f'\n{"="*60}')
    print(f'Benchmarking: {case_name.upper()}')
    print(f'{"="*60}')
    
    set_case_config(case_name)
    
    cfg = CASE_CONFIGS[case_name]
    sample_length = cfg['sample_length']
    feature_size = cfg['feature_size']
    input_shape_ts = (1, sample_length, feature_size)  # (B, S, F) for time-series models
    n_bus = cfg['n_bus']
    input_shape_graph = (1, n_bus, 2)  # (B, N, 2) for GraphAE
    
    # ── Profile OCSVM ──
    res = profile_ocsvm(case_name)
    if res:
        all_results[(case_name, 'OCSVM')] = res
    
    # ── Profile ConvAE ──
    from models.baselines.convae import Conv_AE
    res = profile_torch_model(Conv_AE, 'ConvAE', input_shape_ts, device)
    if res:
        all_results[(case_name, 'ConvAE')] = res
    
    # ── Profile LSTMAE ──
    from models.baselines.lstmae import LSTM_AE
    res = profile_torch_model(LSTM_AE, 'LSTMAE', input_shape_ts, device)
    if res:
        all_results[(case_name, 'LSTMAE')] = res
    
    # ── Profile GraphAE ──
    from models.baselines.graph_ae import GraphAE
    res = profile_torch_model(GraphAE, 'GraphAE', input_shape_graph, device)
    if res:
        all_results[(case_name, 'GraphAE')] = res
    
    # ── Profile AT (Anomaly Transformer) ──
    from models.baselines.anomaly_transformer import AnomTrans
    res = profile_torch_model(AnomTrans, 'AT', input_shape_ts, device)
    if res:
        all_results[(case_name, 'AT')] = res
    
    # ── Profile PIConv (PIConvAE) ──
    from models.baselines.piconvae import PIConvAE
    res = profile_torch_model(PIConvAE, 'PIConv', input_shape_ts, device)
    if res:
        all_results[(case_name, 'PIConv')] = res
    
    # ── Profile PGAE-tau ──
    from models.detection.pgae_tau import PGAE_tau
    res = profile_torch_model(PGAE_tau, 'PGAE-tau', input_shape_ts, device)
    if res:
        all_results[(case_name, 'PGAE-tau')] = res

print(f'\n{"="*60}')
print(f'Benchmark complete.  Collected {len(all_results)} data points.')


In [ ]:
# ── Build display DataFrame ──
rows = []
for case_name in ['case14', 'case30', 'case57']:
    for model_name in MODEL_NAMES_DISPLAY:
        key = (case_name, model_name)
        if key in all_results:
            r = all_results[key]
            rows.append({
                'Case': case_name,
                'Model': model_name,
                'Params (M)': r['Total Params (M)'],
                'Inference Time (ms)': r['Inference Time (ms)'],
            })

df = pd.DataFrame(rows)

# Pivot for display
print('\n' + '='*80)
print('PERFORMANCE METRICS SUMMARY')
print('='*80)

for case_name in ['case14', 'case30', 'case57']:
    print(f'\n--- {case_name.upper()} ---')
    sub = df[df['Case'] == case_name][['Model', 'Params (M)', 'Inference Time (ms)']]
    sub = sub.set_index('Model')
    display(sub)

# ── Write to Excel ──
from configs.paths import OUTPUT_ROOT
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
save_path = OUTPUT_ROOT / 'inference_profile.xlsx'
wb = Workbook()
wb.remove(wb.active)

# Remove existing 性能指标 sheet if present, create fresh
if '性能指标' in wb.sheetnames:
    del wb['性能指标']

ws = wb.create_sheet('性能指标')

# ── Styles ──
header_font = Font(name='Times New Roman', bold=True, size=11)
title_font = Font(name='Times New Roman', bold=True, size=12)
data_font = Font(name='Times New Roman', size=11)
center_align = Alignment(horizontal='center', vertical='center')
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
header_fill = PatternFill(start_color='D9E1F2', end_color='D9E1F2', fill_type='solid')

# ── Row 1: Title ──
ws.merge_cells('A1:G1')
ws['A1'] = '模型推理性能对比（推理速度与计算量）'
ws['A1'].font = title_font
ws['A1'].alignment = center_align

# ── Row 2: Case headers ──
ws.merge_cells('B2:C2')
ws['B2'] = 'Case-14'
ws.merge_cells('D2:E2')
ws['D2'] = 'Case-30'
ws.merge_cells('F2:G2')
ws['F2'] = 'Case-57'
for col in ['B', 'D', 'F']:
    ws[f'{col}2'].font = header_font
    ws[f'{col}2'].alignment = center_align
    ws[f'{col}2'].fill = header_fill
    ws[f'{col}2'].border = thin_border

# ── Row 3: Metric sub-headers ──
sub_headers = {
    'A3': 'Method',
    'B3': '参数量(M)', 'C3': '推理时间(ms)',
    'D3': '参数量(M)', 'E3': '推理时间(ms)',
    'F3': '参数量(M)', 'G3': '推理时间(ms)',
}
for cell_ref, text in sub_headers.items():
    ws[cell_ref] = text
    ws[cell_ref].font = header_font
    ws[cell_ref].alignment = center_align
    ws[cell_ref].fill = header_fill
    ws[cell_ref].border = thin_border

# ── Rows 4-10: Data ──
case_col_map = {
    'case14': ('B', 'C'),
    'case30': ('D', 'E'),
    'case57': ('F', 'G'),
}

for row_idx, model_name in enumerate(MODEL_NAMES_DISPLAY):
    excel_row = row_idx + 4
    
    # Method name
    ws.cell(row=excel_row, column=1, value=model_name)
    ws.cell(row=excel_row, column=1).font = data_font
    ws.cell(row=excel_row, column=1).alignment = center_align
    ws.cell(row=excel_row, column=1).border = thin_border
    
    for case_name in ['case14', 'case30', 'case57']:
        key = (case_name, model_name)
        params_col, time_col = case_col_map[case_name]
        
        if key in all_results:
            r = all_results[key]
            params_val = r['Total Params (M)']
            time_val = r['Inference Time (ms)']
            
            # Params
            cell_p = ws.cell(row=excel_row, column=ord(params_col) - ord('A') + 1, value=round(params_val, 4))
            cell_p.font = data_font
            cell_p.alignment = center_align
            cell_p.border = thin_border
            
            # Time
            cell_t = ws.cell(row=excel_row, column=ord(time_col) - ord('A') + 1, value=round(time_val, 4))
            cell_t.font = data_font
            cell_t.alignment = center_align
            cell_t.border = thin_border
        else:
            for col_letter in [params_col, time_col]:
                col_idx = ord(col_letter) - ord('A') + 1
                cell = ws.cell(row=excel_row, column=col_idx, value='N/A')
                cell.font = data_font
                cell.alignment = center_align
                cell.border = thin_border

# ── Add OCSVM footnote ──
note_row = len(MODEL_NAMES_DISPLAY) + 5
ws.merge_cells(f'A{note_row}:G{note_row}')
note_font = Font(name='Times New Roman', size=10, italic=True)
ws.cell(row=note_row, column=1, 
        value='注：OCSVM的"参数量"列数值为支持向量（Support Vectors）个数，非百万参数量。')
ws.cell(row=note_row, column=1).font = note_font
ws.cell(row=note_row, column=1).alignment = Alignment(horizontal='left', vertical='center')

# ── Column widths ──
ws.column_dimensions['A'].width = 14
for col_letter in ['B', 'C', 'D', 'E', 'F', 'G']:
    ws.column_dimensions[col_letter].width = 16

# ── Save ──
wb.save(save_path)
print(f'\nResults written to: {save_path}  →  Sheet: 性能指标')
